In [285]:
import torch
import matplotlib.pyplot as plt
import pandas as pd

In [286]:
mens = open('data/men.txt').read().splitlines()
womens = open('data/women.txt').read().splitlines()

df = pd.DataFrame(mens + womens, columns=['name'])

df['name'] = df['name'].str.lower()
df = df[~df['name'].str.endswith('.')]
df = df[df['name'].str.len() >= 3]
df = df.drop_duplicates().reset_index(drop=True)
df = df.sample(frac=1.0).reset_index(drop=True) # shuffle

In [287]:
df.describe()

,name
count,23184
unique,23184
top,abdilahi
freq,1


In [288]:
names = df['name'].to_list()
names[:10]

['abdilahi',
 'ragnarsson',
 'tarek',
 'brenda',
 'gerli',
 'ajsa',
 'robertsdotter',
 'sissel',
 'ottoville',
 'jasmir']

In [289]:
vocabulary = sorted(list(set(''.join(names))))
vocabulary.insert(0, '<S>')
vocabulary.insert(1, '<E>')
len(vocabulary)

54

In [290]:
atoi = {c: i for i, c in enumerate(vocabulary)}
itoa = {i: c for i, c in enumerate(vocabulary)}
atoi

{'<S>': 0,
 '<E>': 1,
 "'": 2,
 '-': 3,
 'a': 4,
 'b': 5,
 'c': 6,
 'd': 7,
 'e': 8,
 'f': 9,
 'g': 10,
 'h': 11,
 'i': 12,
 'j': 13,
 'k': 14,
 'l': 15,
 'm': 16,
 'n': 17,
 'o': 18,
 'p': 19,
 'q': 20,
 'r': 21,
 's': 22,
 't': 23,
 'u': 24,
 'v': 25,
 'w': 26,
 'x': 27,
 'y': 28,
 'z': 29,
 'à': 30,
 'á': 31,
 'â': 32,
 'ã': 33,
 'ä': 34,
 'å': 35,
 'ç': 36,
 'è': 37,
 'é': 38,
 'ê': 39,
 'ë': 40,
 'í': 41,
 'î': 42,
 'ï': 43,
 'ò': 44,
 'ó': 45,
 'ô': 46,
 'õ': 47,
 'ö': 48,
 'ø': 49,
 'ù': 50,
 'ú': 51,
 'ü': 52,
 'þ': 53}

In [291]:
block_size = 8 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w:
      ix = atoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append
    context = context[1:] + [1] # add <E> at the end of the word

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(names))
n2 = int(0.9*len(names))
Xtr,  Ytr  = build_dataset(names[:n1])     # 80%
Xdev, Ydev = build_dataset(names[n1:n2])   # 10%
Xte,  Yte  = build_dataset(names[n2:])     # 10%

torch.Size([128357, 8]) torch.Size([128357])
torch.Size([15760, 8]) torch.Size([15760])
torch.Size([15956, 8]) torch.Size([15956])


In [292]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itoa[int(ix.item())] for ix in x), '-->', itoa[int(y.item())])

<S><S><S><S><S><S><S><S> --> a
<S><S><S><S><S><S><S>a --> b
<S><S><S><S><S><S>ab --> d
<S><S><S><S><S>abd --> i
<S><S><S><S>abdi --> l
<S><S><S>abdil --> a
<S><S>abdila --> h
<S>abdilah --> i
<S><S><S><S><S><S><S><S> --> r
<S><S><S><S><S><S><S>r --> a
<S><S><S><S><S><S>ra --> g
<S><S><S><S><S>rag --> n
<S><S><S><S>ragn --> a
<S><S><S>ragna --> r
<S><S>ragnar --> s
<S>ragnars --> s
ragnarss --> o
agnarsso --> n
<S><S><S><S><S><S><S><S> --> t
<S><S><S><S><S><S><S>t --> a


In [293]:
vocab_size = len(vocabulary)
n_embd = 24
n_hidden = 128

In [312]:
class WaveNet(torch.nn.Module):
  def __init__(self, vocab_size, n_embd, n_hidden):
    super().__init__()
    self.n = 2
    self.embedding = torch.nn.Embedding(vocab_size, n_embd)
    self.linear1 = torch.nn.Linear(n_embd * 2, n_hidden, bias=False)
    self.bn1 = torch.nn.BatchNorm1d(n_hidden*4)
    self.linear2 = torch.nn.Linear(n_hidden*2, n_hidden, bias=False)
    self.bn2 = torch.nn.BatchNorm1d(n_hidden*2)
    self.linear3 = torch.nn.Linear(n_hidden*2, n_hidden, bias=False)
    self.bn3 = torch.nn.BatchNorm1d(n_hidden*4)
    self.linear4 = torch.nn.Linear(n_hidden, vocab_size)

  def forward(self, x):
    x = self.embedding(x)

    x = self._flatten_consecutive(x)
    x = self.linear1(x)
    x = self._batchnorm(x, self.bn1)
    x = torch.tanh(x)

    x = self._flatten_consecutive(x)
    x = self.linear2(x)
    x = self._batchnorm(x, self.bn2)
    x = torch.tanh(x)

    x = self._flatten_consecutive(x)
    x = self.linear3(x)
    x = self._batchnorm(x, self.bn3)
    x = torch.tanh(x)

    logits = self.linear4(x)

    return logits
  
  def _batchnorm(self, x, bn):
    start_shape = x.shape

    if len(x.shape) == 3:
      _, T, C = x.shape
      x = bn(x.view(-1, T*C))

    x = x.view(start_shape)

    return x
  
  def _flatten_consecutive(self, x):
    B, T, C = x.shape
    x = x.view(B, T//self.n, C*self.n)
    if x.shape[1] == 1:
      x = x.squeeze(1)
    
    return x

In [313]:
model = WaveNet(vocab_size, n_embd, n_hidden)

In [314]:
model(Xtr[:5])

tensor([[-0.1350,  0.3396, -0.3098,  0.1245,  0.0305,  0.0688, -0.1325, -0.0132,
         -0.1110,  0.0400, -0.0154, -0.1000, -0.3022, -0.1194,  0.0607, -0.0898,
         -0.2597, -0.0926,  0.1412,  0.0858,  0.0149,  0.1971, -0.0461,  0.1651,
         -0.2116,  0.0608, -0.0823, -0.0478, -0.0440,  0.3115,  0.0203,  0.0796,
          0.0494,  0.2017,  0.2996, -0.1261,  0.0930,  0.2862, -0.0020,  0.0503,
          0.0488,  0.3362, -0.1229, -0.1370, -0.3346,  0.2078,  0.0977,  0.0888,
          0.0856, -0.1778, -0.0146,  0.0723, -0.0424, -0.2010],
        [-0.1060,  0.1212,  0.0856, -0.1028,  0.1160, -0.0423,  0.0239, -0.2180,
          0.0147,  0.1870, -0.0485,  0.1845, -0.1105, -0.0434,  0.2474, -0.2111,
          0.0481, -0.0292,  0.1148, -0.0497, -0.0559,  0.1488, -0.0451,  0.2123,
         -0.2917, -0.0827,  0.0160,  0.3045, -0.0557,  0.0453,  0.1074,  0.0112,
          0.0972,  0.1690,  0.1978,  0.1793, -0.0472, -0.0620, -0.0599,  0.1173,
          0.1524,  0.1253,  0.2076,  0.2286, 

In [315]:
from torch.nn import functional as F

probs = F.softmax(model(Xtr[:5]), dim=1)
pred = probs.max(dim=1).indices
pred

tensor([ 1, 27, 30, 49, 31])

In [316]:
for p in pred:
  print(itoa[int(p.item())])

<E>
x
à
ø
á


In [317]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [318]:
count_parameters(model)

82502